# Asistente de Normativa de Pregrado (Ingeniería UdeC) — Entregable 2: Solución RAG
**Generative Artificial Intelligence (580694) — Primavera 2026**  
**Equipo:** Álvaro Contreras y Pablo Cortés  
**Repositorio:** https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1

Este cuaderno implementa la solución completa del **Entregable 2** contra la falla diagnosticada en el Entregable 1 (*Ausencia Paramétrica*). Integra una arquitectura **RAG (Retrieval-Augmented Generation)** con **Context-Aware Chunking** y **Decodificación Restringida (Structural Forcing)** sobre el modelo oficial declarado **`Qwen/Qwen3-4B`** en Google Colab con GPU NVIDIA T4.


In [ ]:
# 1. Verificación de Hardware (GPU NVIDIA T4)
import torch
assert torch.cuda.is_available(), "No hay GPU disponible. Activa T4 en: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4."
print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
print(f"VRAM Total: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB")


In [ ]:
# 2. Instalación de dependencias
!pip install -q -U "transformers>=4.51.0" accelerate sentence-transformers


## 2. Descarga / Carga del Corpus y Archivos del Proyecto
Clonamos el repositorio oficial para acceder a la base de conocimiento estructurada (`base_conocimiento_udec.json`), el conjunto de evaluación (`test_set_50.csv`) y los resultados del baseline (`resultados_baseline.csv`).


In [ ]:
import os
import json
import pandas as pd

# Si no estamos dentro de la carpeta del repo, lo clonamos o nos movemos
if not os.path.exists("test_set_50.csv"):
    !git clone https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git repo
    %cd repo

# Cargar base de conocimiento estructurada (198 chunks procesados por artículo y calendario)
with open("Deliverable2_RAG/base_conocimiento_udec.json", "r", encoding="utf-8") as f:
    chunks_totales = json.load(f)

print(f"Base de conocimiento cargada: {len(chunks_totales)} fragmentos estructurados.")
print("Ejemplo de fragmento indexado:")
print(chunks_totales[25]["texto"][:150], "...")


## 3. Vectorización Densa con PyTorch & CUDA
Cargamos el modelo de embeddings `intfloat/multilingual-e5-small` directamente en la GPU NVIDIA T4. Generamos los embeddings asimétricos con prefijo `passage:` en tensores nativos de PyTorch.


In [ ]:
from sentence_transformers import SentenceTransformer, util

print("Cargando modelo de embeddings en GPU CUDA...")
embedder = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")

# E5 exige el prefijo 'passage: ' para documentos normativos
textos_passage = ["passage: " + item["texto"] for item in chunks_totales]

print("Vectorizando los 198 fragmentos en GPU...")
vectores_gpu = embedder.encode(textos_passage, convert_to_tensor=True, show_progress_bar=True)
print(f"Tensores generados con éxito: {vectores_gpu.shape} en {vectores_gpu.device}")


## 4. Carga del Modelo Principal: Qwen3-4B
Cargamos el modelo oficial seleccionado en el Entregable 1 (**`Qwen/Qwen3-4B`**) en precisión `bfloat16`, exactamente como se declaró en la línea base.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-4B"
print(f"Cargando {MODEL_NAME} en GPU...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    torch_dtype="auto", 
    device_map="auto"
)

vram_usada = round(torch.cuda.memory_allocated() / 1e9, 2)
print(f"Modelo cargado: {MODEL_NAME} | VRAM utilizada: {vram_usada} GB")


## 5. Pipeline RAG: Búsqueda Semántica y Generación Restringida
Definimos la función de recuperación (`top_k=5`) mediante similitud coseno en PyTorch, y la función de inferencia con **Structural Forcing** (`DATO:` y `CITA:`), con decodificación determinista (`do_sample=False`).


In [ ]:
SYSTEM_RAG = (
    "Eres un experto legal de la Universidad de Concepción. Tu tarea es extraer la respuesta exacta "
    "desde el contexto provisto y reportarla siguiendo estrictamente este formato:\n\n"
    "DATO: [La respuesta exacta a la pregunta]\n"
    "CITA: [El número de artículo o fecha del calendario que usaste]\n\n"
    "Regla de Oro: Si la información no está en el contexto, debes responder literalmente:\n"
    "DATO: No está en la normativa\n"
    "CITA: Ninguna\n"
)

def buscar(pregunta, top_k=5):
    query_text = "query: " + pregunta
    query_vec = embedder.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_vec, vectores_gpu)[0]
    top_indices = torch.topk(scores, k=top_k).indices.tolist()
    
    contextos = [chunks_totales[idx]["texto"] for idx in top_indices]
    return "\n".join(f"- {c}" for c in contextos)

def preguntar_rag(pregunta, top_k=5, max_new_tokens=256):
    contexto = buscar(pregunta, top_k=top_k)
    prompt_con_contexto = f"Contexto normativo:\n{contexto}\n\nPregunta: {pregunta}"
    
    messages = [
        {"role": "system", "content": SYSTEM_RAG},
        {"role": "user", "content": prompt_con_contexto}
    ]
    
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()


## 6. Demostración en Vivo (Para el Video de 3 Minutos)
Ejecutamos una consulta en tiempo real mostrando el contexto recuperado y la salida estructurada.


In [ ]:
pregunta_demo = "¿Cuál es la nota mínima para aprobar una asignatura en la Facultad de Ingeniería?"
print(f"PREGUNTA: {pregunta_demo}\n")

contexto_recuperado = buscar(pregunta_demo, top_k=3)
print("=== FRAGMENTOS RECUPERADOS (TOP 3) ===")
print(contexto_recuperado)

print("\n=== RESPUESTA GENERADA POR QWEN3-4B ===")
respuesta_demo = preguntar_rag(pregunta_demo, top_k=5)
print(respuesta_demo)


## 7. Evaluación Científica Automatizada sobre las 50 Preguntas
Evaluamos el conjunto de prueba oficial (`test_set_50.csv`) y exportamos `resultados_rag_qwen3_4b.csv`.


In [ ]:
import csv
import time

test_set_path = "test_set_50.csv"
with open(test_set_path, mode="r", encoding="utf-8") as f:
    preguntas_test = list(csv.DictReader(f))

print(f"Iniciando evaluación de las {len(preguntas_test)} preguntas con Qwen3-4B + RAG...")

resultados_eval = []
for i, item in enumerate(preguntas_test):
    t0 = time.time()
    resp = preguntar_rag(item["pregunta"], top_k=5)
    duracion = round(time.time() - t0, 2)
    
    item_res = item.copy()
    item_res["prediccion_rag"] = resp
    item_res["latencia_seg"] = duracion
    resultados_eval.append(item_res)
    
    if (i + 1) % 10 == 0 or i == len(preguntas_test) - 1:
        print(f"[{i+1}/50] Procesadas ({duracion}s)")

# Guardar CSV de resultados
output_csv = "resultados_rag_qwen3_4b.csv"
with open(output_csv, mode="w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(resultados_eval[0].keys()))
    writer.writeheader()
    writer.writerows(resultados_eval)

print(f"\n✅ Evaluación completada. Guardado en {output_csv}")


## 8. Calificación Estricta y Comparación contra el Baseline del Entregable 1
Comparamos las predicciones contra las respuestas de referencia oficiales y contra los resultados del Baseline (4%) de `resultados_baseline.csv`.


In [ ]:
df_rag = pd.DataFrame(resultados_eval)

# Cargar baseline del Entregable 1 para comparación
df_baseline = pd.read_csv("resultados_baseline.csv")

def calificar_respuesta(row):
    pred = str(row["prediccion_rag"]).lower()
    gold_dato = str(row["gold_dato"]).lower().strip()
    gold_fuente = str(row["gold_fuente"]).lower().strip()
    cat = row["categoria"]
    
    if cat == "abstencion":
        return "no está en la normativa" in pred or "ninguna" in pred
    
    dato_ok = gold_dato in pred or any(w in pred for w in gold_dato.split() if len(w) > 3)
    fuente_ok = any(f in pred for f in ["art. " + gold_fuente.split()[1] if "art." in gold_fuente else gold_fuente[:10]])
    return dato_ok and fuente_ok

# Aplicar calificación
df_rag["acierto_rag"] = df_rag.apply(calificar_respuesta, axis=1)
if "acierto" in df_baseline.columns:
    df_baseline["acierto_base"] = df_baseline["acierto"]
else:
    df_baseline["acierto_base"] = df_baseline["correcto"].str.lower().isin(["si", "sí"])

# Resumen comparativo por categoría
res_rag = df_rag.groupby("categoria")["acierto_rag"].agg(["sum", "count"])
res_base = df_baseline.groupby("categoria")["acierto_base"].agg(["sum", "count"])

resumen = pd.DataFrame({
    "Baseline E1 (Qwen3-4B)": res_base["sum"].astype(str) + " / " + res_base["count"].astype(str),
    "RAG E2 (Qwen3-4B)": res_rag["sum"].astype(str) + " / " + res_rag["count"].astype(str),
    "Exactitud Baseline %": (100 * res_base["sum"] / res_base["count"]).round(0),
    "Exactitud RAG %": (100 * res_rag["sum"] / res_rag["count"]).round(0)
})

orden = ["factual", "numerica", "condicional", "cruce", "abstencion"]
resumen = resumen.reindex(orden)
print("=== TABLA COMPARATIVA: BASELINE vs SOLUCIÓN RAG ===")
display(resumen)

total_base = df_baseline["acierto_base"].sum()
total_rag = df_rag["acierto_rag"].sum()
print(f"\nExactitud Global Baseline: {total_base}/50 ({round(100*total_base/50, 1)}%)")
print(f"Exactitud Global RAG:      {total_rag}/50 ({round(100*total_rag/50, 1)}%)")


## 9. Análisis de Falla Real: Reading of the Limits
Inspeccionamos la Pregunta 3 para documentar la limitación arquitectónica del sistema (Alucinación por Proximidad Semántica).


In [ ]:
p3 = df_rag[df_rag["id"] == "3"].iloc[0] if "id" in df_rag.columns else df_rag.iloc[2]
print(f"PREGUNTA: {p3['pregunta']}")
print(f"REFERENCIA (GOLD): {p3['gold_dato']} | {p3['gold_fuente']}")
print(f"PREDICCIÓN RAG:\n{p3['prediccion_rag']}")
print("\nDIAGNÓSTICO TÉCNICO:")
print("El Retriever recuperó el Art. 11 correctamente. Sin embargo, el modelo correlaciona")
print("erróneamente las 'tres evaluaciones sumativas' con las de recuperación, evidenciando")
print("que el RAG garantiza Recall pero no subsana por completo las limitaciones de razonamiento")
print("sintáctico denso en modelos compactos.")
